# 02 Polar Verity Sense Offline Runbook

## A. Setup

In [5]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_ROOT = Path.cwd()
LIB_ROOT = NOTEBOOK_ROOT / "lib"
if str(LIB_ROOT) not in sys.path:
    sys.path.insert(0, str(LIB_ROOT))

from session_discovery import list_sessions
from remote_sync import get_session_by_id
from pipeline_client import run_full_pipeline
from session_loader import load_summary, load_clean, load_window_features, load_raw_chunks
from eda_helpers import compute_gap_stats, ensure_datetime, pick_first_column

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

In [6]:
# Runtime parameters
user_id = 1
session_id_override = "f20f5ed7-55e2-46d1-99b5-12368729a254"  # set specific session id if needed
user_id_candidates = [1, 2]  # fallback search when session_id_override is set
max_rows_per_stream = 100000
env_path = NOTEBOOK_ROOT / ".env"
plot_points = 4000
live_progress = False  # avoid SSH progress polling loop issues

## B. Session Discovery

In [7]:
sessions = list_sessions(user_id=user_id, env_path=env_path)
offline_sessions = [s for s in sessions if str(s.get("source","")).lower() == "polar_verity_sense" and str(s.get("collection_mode","")).lower() == "offline_recording"]
print(f"discovered={len(sessions)} offline_verity={len(offline_sessions)}")
pd.DataFrame(offline_sessions).head(20)

discovered=0 offline_verity=0


""


In [8]:
if session_id_override:
    # Resolve correct user_id for explicit session_id override.
    matched_user_id = None
    for candidate_user_id in user_id_candidates:
        try:
            candidate_sessions = list_sessions(user_id=candidate_user_id, env_path=env_path)
        except Exception:
            continue
        if any(str(item.get("session_id")) == str(session_id_override) for item in candidate_sessions):
            matched_user_id = candidate_user_id
            break

    if matched_user_id is not None:
        user_id = matched_user_id
    session_id = session_id_override
elif offline_sessions:
    session_id = str(offline_sessions[0]["session_id"])
else:
    raise RuntimeError("No Polar Verity Sense offline sessions found. Set session_id_override.")

print("user_id:", user_id)
print("session_id:", session_id)


session_id: f20f5ed7-55e2-46d1-99b5-12368729a254


## C. Run Pipeline

In [ ]:
pipeline_result = run_full_pipeline(
    session_id=session_id,
    user_id=user_id,
    env_path=env_path,
    live_progress=False,
)
pipeline_result

KeyboardInterrupt: 

pipeline_http_trigger_failed
ssh_command_failed


## D. Sync Data

In [ ]:
session_cache_path = get_session_by_id(session_id=session_id, env_path=env_path)
print(session_cache_path)

import json
## E. Load Artifacts

In [ ]:
summary = load_summary(session_cache_path)
clean = load_clean(session_cache_path, max_rows_per_stream=max_rows_per_stream)
window_features = load_window_features(session_cache_path, max_rows_per_stream=max_rows_per_stream)
raw = load_raw_chunks(session_cache_path, preview_rows_per_stream=20, max_scan_lines_per_stream=5000)

print("clean streams:", sorted(clean.keys()))
print("window streams:", sorted(window_features.keys()))
print("raw streams:", sorted(raw.keys()))
summary.get("_source_path")

## F. Time Alignment Reports

In [ ]:
report_paths = sorted(Path(session_cache_path).glob("**/processed/clean_timeseries/**/time_alignment_report.json"))
reports = []
for p in report_paths:
    payload = json.loads(p.read_text(encoding="utf-8"))
    payload["_path"] = str(p)
    reports.append(payload)

report_df = pd.DataFrame([
    {
        "stream_type": r.get("stream_type"),
        "alignment_basis": r.get("alignment_basis"),
        "epoch_offset_decision": r.get("epoch_offset_decision"),
        "confidence": r.get("confidence"),
        "samples_count": r.get("samples_count"),
        "start_utc": (r.get("normalized_time_range") or {}).get("start_utc"),
        "end_utc": (r.get("normalized_time_range") or {}).get("end_utc"),
        "warnings": "; ".join(r.get("warnings") or []),
        "path": r.get("_path"),
    }
    for r in reports
])
report_df.sort_values(["stream_type"]).reset_index(drop=True) if not report_df.empty else report_df

## G. Stream Quality Checks

In [ ]:
rows=[]
for stream, df in clean.items():
    local, ts_col = ensure_datetime(df)
    if ts_col is None or local.empty:
        rows.append({"stream":stream,"rows":len(df.index),"ts_col":ts_col,"duration_s":None,"median_delta_s":None,"max_delta_s":None,"duplicates":None})
        continue
    gaps = compute_gap_stats(local[ts_col])
    duration = (local[ts_col].max() - local[ts_col].min()).total_seconds() if len(local.index)>1 else 0
    rows.append({
        "stream":stream,
        "rows":len(local.index),
        "ts_col":ts_col,
        "duration_s":float(duration),
        "median_delta_s":gaps.median_delta_seconds,
        "max_delta_s":gaps.max_delta_seconds,
        "duplicates":gaps.duplicate_timestamps,
    })

pd.DataFrame(rows).sort_values("stream")

## H. Stream Tables And Plots

In [ ]:
plot_columns = {
    "hr": ["hr", "corrected_hr"],
    "ppi": ["pp_in_ms", "hr"],
    "acc": ["x", "y", "z", "vector_magnitude"],
    "gyro": ["x", "y", "z", "vector_magnitude"],
    "mag": ["x", "y", "z", "vector_magnitude"],
    "ppg": ["ppg0", "ppg1", "ppg2", "ambient"],
}
for stream, df in sorted(clean.items()):
    print(f"\n### {stream} ###")
    if df.empty:
        print("empty")
        continue
    local, ts_col = ensure_datetime(df)
    display(local.head(10))
    if ts_col is None:
        print("no timestamp column")
        continue
    cols = [c for c in plot_columns.get(stream, []) if c in local.columns]
    if not cols:
        print("no plot columns")
        continue
    sampled = local if len(local.index) <= plot_points else local.iloc[::max(1, len(local.index)//plot_points)].copy()
    ax = sampled.plot(x=ts_col, y=cols, figsize=(12, 3), title=f"{stream} samples")
    ax.set_xlabel("ts_utc")
    plt.show()

## I. Session Summary Snapshot

In [ ]:
if summary:
    print("status:", summary.get("status"))
    print("streams_present:", summary.get("streams_present"))
    print("streams_missing:", summary.get("streams_missing"))
    display(pd.DataFrame(summary.get("stream_summaries", {})).T[["status","warnings"]])
else:
    print("summary missing")